# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window


One row represents the daily performance of one content page for one client.

The grain of the fact_content_daily_performance table is:

report_date + client_hash_id + content_hash_id

This notebook will use a middle month (March 2026) for feature development, following the project guidance.

The prediction task is to rank pages by refresh opportunity using historical search and engagement signals.

Future information is excluded to prevent data leakage.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")

login(token=hf_token)

from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

print(dataset["train"].features)
print(dataset["train"][0])

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

## 2. Fields: feature / label / context / excluded

## Features

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_engaged_sessions
- scroll_events

These are historical observations available before a refresh decision.

## Label / Proxy

There is no direct "Needs Refresh" label in the dataset.

A proxy target will be defined later using future observed outcomes.
## Context

- report_date
- client_hash_id
- content_hash_id

These identify observations but are not predictive features.

## Excluded

- client_has_gsc
- client_has_ga4

These indicate tracking availability rather than page quality.

Future-derived metrics are excluded because they would leak target information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
train = dataset["train"]

print("Total rows:", len(train))

print("\nColumns:")
print(train.features)

print("\nExample row:")
print(train[0])

#grain verification
row = train[0]

print("Report Date :", row["report_date"])
print("Client ID   :", row["client_hash_id"])
print("Content ID  :", row["content_hash_id"])

# availability
sample = train.select(range(10000))

gsc_rows = sum(r["gsc_data_available"] for r in sample)
ga4_rows = sum(r["ga4_data_available"] for r in sample)

print("Rows with GSC:", gsc_rows)
print("Rows with GA4:", ga4_rows)

# date window
dates = [r["report_date"] for r in train.select(range(10000))]

print("Earliest:", min(dates))
print("Latest:", max(dates))

Total rows: 78835655

Columns:
{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int6

## 4. Data limits


This dataset contains historical observations from Google Search Console and Google Analytics 4.

It cannot prove that refreshing a page causes better rankings or higher traffic.

Search performance is influenced by seasonality, competitors, search engine changes, and user behaviour.

The warehouse is an unbalanced panel because different clients started tracking at different times. Rows before a client's GA4 start contain search data only, so availability flags must be considered when defining feature windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.